# Web Data Collection IV: Dynamic Pages and Wrap-up

**MGS 4701 W01** | Week 4-1 | Tuesday, 22 September 2026

| § | | Min |
| --- | --- | --- |
| 0 | Review of last session | 5 |
| 1 | Browser automation | 15 |
| | **🖐 Exercise** | 12 |
| 2 | Anti-scraping | 6 |
| 3 | Playwright and Scrapy | 8 |
| 4 | Pagination | 10 |
| 5 | The whole project, as pseudo-code | 6 |
| 6 | Web data collection wrap-up | 10 |

---
# 0. Where we are

**Last Tuesday:**

- An **API URL** returns JSON; an **ordinary URL** returns HTML written for display.
- **BeautifulSoup** turns HTML into something searchable — `find`, `select`, `get_text` — and a regex on the *text* often survives redesigns better than a
  CSS class.
- **Links inside a page**: collect by URL pattern, and always set a stopping rule.
- **Dynamic pages**: `/scroll` returned `200 OK` and zero quotes (e.g., <https://quotes.toscrape.com/scroll>). DevTools found the page's own API, and we called that instead.

We did not reach Selenium. That is where today starts.

> ⏰ **A1 — project brief and pilot data — is due Monday 28 September.**

*Pilot data* means a small first sample, collected with the method you intend to use at full scale. Its job is to prove the method works — and to show you what breaks — before you commit to it.

In [1]:
import re
import time
import xml.etree.ElementTree as ET
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "MGS4701 teaching exercise (dchen@kean.edu)"}
print("ready")

ready


---
# 1. Browser automation — when there is no door

When a URL returns an <span style="color:red">empty shell</span> — `requests.get()`
succeeds, but the data you can see on screen is not in the HTML — check in this order:

| Step | Where | Example | If you find it |
| --- | --- | --- | --- |
| 1 | DevTools → Network → **Fetch/XHR** → reload | a company careers page loading jobs from `boards-api.greenhouse.io` | call that URL directly with `requests.get()` |
| 2 | Same panel, a POST to **`/graphql`** | GitHub (documented) · LeetCode (**disallowed** in robots.txt — look, don't call) | copy the query and variables from the Payload tab into `requests.post()` |
| 3 | Nothing usable, or the data only appears after you **run JavaScript** or **interact** | `quotes.toscrape.com/js/`; a "Load more" button | **browser automation** |

**What "run JavaScript" means:** JavaScript is a programming language that runs **inside your browser**. On a dynamic page the server sends two things: an HTML skeleton, and a script. Your browser then *executes* the script — runs it — and the script builds the content and inserts it into the page.

`requests` downloads the script as text but **never runs it**. You receive the recipe, not the dish.

In [2]:
r = requests.get("https://quotes.toscrape.com/js/", headers=HEADERS, timeout=15)
s = BeautifulSoup(r.text, "html.parser")
scripts = s.find_all("script")

print("quote elements in the HTML :", len(s.select(".quote")))
print("<script> blocks            :", len(scripts))

biggest = max(scripts, key=lambda t: len(t.text), default=None)
print("\nThe largest script begins:\n")
print(biggest.text.strip()[:300] if biggest else "none")

quote elements in the HTML : 0
<script> blocks            : 2

The largest script begins:

var data = [
    {
        "tags": [
            "change",
            "deep-thoughts",
            "thinking",
            "world"
        ],
        "author": {
            "name": "Albert Einstein",
            "goodreads_link": "/author/show/9810.Albert_Einstein",
            "slug": "Albert-Ein


In [3]:
s

<!DOCTYPE html>

<html lang="en">
<head>
<meta charset="utf-8"/>
<title>Quotes to Scrape</title>
<link href="/static/bootstrap.min.css" rel="stylesheet"/>
<link href="/static/main.css" rel="stylesheet"/>
</head>
<body>
<div class="container">
<div class="row header-box">
<div class="col-md-8">
<h1>
<a href="/" style="text-decoration: none">Quotes to Scrape</a>
</h1>
</div>
<div class="col-md-4">
<p>
<a href="/login">Login</a>
</p>
</div>
</div>
<script src="/static/jquery.js"></script>
<script>
    var data = [
    {
        "tags": [
            "change",
            "deep-thoughts",
            "thinking",
            "world"
        ],
        "author": {
            "name": "Albert Einstein",
            "goodreads_link": "/author/show/9810.Albert_Einstein",
            "slug": "Albert-Einstein"
        },
        "text": "\u201cThe world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.\u201d"
    },
    {
        "tags": [
   

Zero quotes in the HTML — but look at the script. That is the recipe. A browser would run it and the quotes would appear; Python just shows you the code.

**Interact:** Some content appears only after someone **does** something: clicks *Load more*, picks a dropdown option, scrolls, types into a search box. No request you can construct by hand reproduces that cleanly.

Both cases need a real browser that runs JavaScript and can be told to click. That is **Selenium**: `pip install selenium`, with Chrome installed. Selenium 4 downloads its own driver on first run.

In [4]:
# DEMO — a real browser, driven by Python
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

opts = Options()
opts.add_argument("--headless=new")          # delete this line to watch it work
driver = webdriver.Chrome(options=opts)

driver.get("https://quotes.toscrape.com/js/")

# Wait until the JavaScript has produced at least one quote — up to 10 seconds
WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, ".quote")))

rendered = BeautifulSoup(driver.page_source, "html.parser")
print("quotes after the browser ran the script:", len(rendered.select(".quote")))

quotes after the browser ran the script: 10


In [ ]:
# rendered
# driver

<selenium.webdriver.chrome.webdriver.WebDriver (session="ddc2237a37408fc0b935f6a16f558eaa")>

`driver.page_source` is the HTML **after** the script ran. From here it is
BeautifulSoup as usual — nothing about parsing changes.

`WebDriverWait` waits for a condition instead of guessing with `time.sleep(2)`.
A fixed sleep is too long on a fast connection and too short on a slow one — and
too short produces an empty page that *looks* like success.

In [6]:
# DEMO — interaction: click "Next", wait for the new page, repeat
rows = []
for page in range(1, 4):                      # three pages, then stop
    for q in BeautifulSoup(driver.page_source, "html.parser").select(".quote"):
        rows.append({"page": page, "author": q.select_one(".author").get_text(strip=True)})

    nxt = driver.find_elements(By.CSS_SELECTOR, "li.next a") # li.next a reads right to left as a description of that structure
    if not nxt:
        break                                  # no Next button: last page
    old = driver.find_element(By.CSS_SELECTOR, ".quote")
    nxt[0].click()                             # the interaction
    WebDriverWait(driver, 10).until(EC.staleness_of(old))

driver.quit()                                  # always close the browser
print(f"{len(rows)} quotes from {rows[-1]['page']} pages")
pd.DataFrame(rows).groupby("page").size()

30 quotes from 3 pages


page
1    10
2    10
3    10
dtype: int64

In [14]:
rows

[{'page': 1, 'author': 'Albert Einstein'},
 {'page': 1, 'author': 'J.K. Rowling'},
 {'page': 1, 'author': 'Albert Einstein'},
 {'page': 1, 'author': 'Jane Austen'},
 {'page': 1, 'author': 'Marilyn Monroe'},
 {'page': 1, 'author': 'Albert Einstein'},
 {'page': 1, 'author': 'André Gide'},
 {'page': 1, 'author': 'Thomas A. Edison'},
 {'page': 1, 'author': 'Eleanor Roosevelt'},
 {'page': 1, 'author': 'Steve Martin'},
 {'page': 2, 'author': 'Marilyn Monroe'},
 {'page': 2, 'author': 'J.K. Rowling'},
 {'page': 2, 'author': 'Albert Einstein'},
 {'page': 2, 'author': 'Bob Marley'},
 {'page': 2, 'author': 'Dr. Seuss'},
 {'page': 2, 'author': 'Douglas Adams'},
 {'page': 2, 'author': 'Elie Wiesel'},
 {'page': 2, 'author': 'Friedrich Nietzsche'},
 {'page': 2, 'author': 'Mark Twain'},
 {'page': 2, 'author': 'Allen Saunders'},
 {'page': 3, 'author': 'Pablo Neruda'},
 {'page': 3, 'author': 'Ralph Waldo Emerson'},
 {'page': 3, 'author': 'Mother Teresa'},
 {'page': 3, 'author': 'Garrison Keillor'},
 {'p

`staleness_of(old)` waits until the old quote has been removed — proof that the
new page actually loaded. Clicking and reading immediately would read the
previous page again.

**The cost.** Every page opens a full browser: roughly a hundred times slower than
`requests`, heavy on memory, and it breaks when Chrome updates. The decision order
does not change: **the page's own API first, browser automation last.**

---
# 🖐 Exercise (12 min)

Using Selenium on `https://quotes.toscrape.com/js/`, collect **author** and
**tags** for every quote on the **first two pages**.

Tags are several `.tag` elements per quote — join them into one string before
they reach the DataFrame.

*If Selenium will not start on your laptop:* use the static twin at
`https://quotes.toscrape.com/`, and write one comment explaining why that is
legitimate here but would not be on a site with no static version.

In [15]:
my_rows = []

# TODO: start a driver and open the page
# TODO: wait for .quote to appear
# TODO: for each .quote, read .author and join the .tag texts
# TODO: click Next once, wait, repeat
# TODO: driver.quit()

for page in range(1, 3):                      # three pages, then stop
    for q in BeautifulSoup(driver.page_source, "html.parser").select(".quote"):
        my_rows.append({"page": page, "author": q.select_one(".author").get_text(strip=True), "tags": ", ".join([t.get_text(strip=True) for t in q.select(".tags .tag")])})

    nxt = driver.find_elements(By.CSS_SELECTOR, "li.next a") # li.next a reads right to left as a description of that structure
    if not nxt:
        break                                  # no Next button: last page
    old = driver.find_element(By.CSS_SELECTOR, ".quote")
    nxt[0].click()                             # the interaction
    WebDriverWait(driver, 10).until(EC.staleness_of(old))

driver.quit()                                  # always close the browser
pd.DataFrame(my_rows).groupby("page").size()

qdf = pd.DataFrame(my_rows)
qdf.head()

MaxRetryError: HTTPConnectionPool(host='localhost', port=51487): Max retries exceeded with url: /session/ddc2237a37408fc0b935f6a16f558eaa/source (Caused by NewConnectionError("HTTPConnection(host='localhost', port=51487): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))

In [9]:
assert len(qdf) > 0, (
    "No rows — the TODO lines in the cell above are still comments. "
    "Fill them in and run that cell again.")
assert len(qdf) == 20, (
    f"Expected 20 quotes (10 per page x 2 pages), got {len(qdf)}. "
    "10 means the Next click or the wait after it did not run.")
assert {"author", "tags"} <= set(qdf.columns), "need author and tags columns"
assert (qdf["author"] != "").all(), "an author is empty — check the selector"
assert not qdf["tags"].apply(lambda x: isinstance(x, list)).any(), \
    "tags must be a joined string, not a list"
print("✓ passed")


AssertionError: No rows — the TODO lines in the cell above are still comments. Fill them in and run that cell again.

<details><summary><b>Solution — open after you have tried</b></summary>

**With Selenium:**

```python
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

opts = Options()
opts.add_argument("--headless=new")
driver = webdriver.Chrome(options=opts)
driver.get("https://quotes.toscrape.com/js/")

my_rows = []
for page in range(1, 3):
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, ".quote")))
    for q in BeautifulSoup(driver.page_source, "html.parser").select(".quote"):
        my_rows.append({
            "author": q.select_one(".author").get_text(strip=True),
            "tags":   "; ".join(t.get_text(strip=True) for t in q.select(".tag")),
        })
    if page < 2:
        old = driver.find_element(By.CSS_SELECTOR, ".quote")
        driver.find_element(By.CSS_SELECTOR, "li.next a").click()
        WebDriverWait(driver, 10).until(EC.staleness_of(old))

driver.quit()
qdf = pd.DataFrame(my_rows)
```

**Without Selenium**, using the static twin:

```python
# Legitimate here because the site publishes the same quotes as static HTML.
# On a site with no static version, this route would not exist.
my_rows, url = [], "https://quotes.toscrape.com/"
for _ in range(2):
    s = BeautifulSoup(requests.get(url, headers=HEADERS, timeout=15).text,
                      "html.parser")
    for q in s.select(".quote"):
        my_rows.append({
            "author": q.select_one(".author").get_text(strip=True),
            "tags":   "; ".join(t.get_text(strip=True) for t in q.select(".tag")),
        })
    url = urljoin(url, s.select_one("li.next a")["href"])
    time.sleep(1)

qdf = pd.DataFrame(my_rows)
```
</details>


---
# 2. Anti-scraping: The walls sites put up

| | What you see | What it means | Usual response |
| --- | --- | --- | --- |
| **Login** | content behind an account | the site limits who sees it | our rule: do not scrape behind a login |
| **Cookies & consent** | a banner over the content | a request for your agreement | clicking it in code accepts terms on your behalf |
| **CAPTCHA** | "verify you are human" | an explicit refusal of automated traffic | **stop** — never solve or outsource one |
| **Rate limit** | `429 Too Many Requests` | you are too fast | slow down; wait longer after each refusal |
| **IP block** | worked, then `403` forever | you were too fast for too long | rotating proxies is evasion, not a fix |
| **Bot detection** | headless browser refused | the site can tell automation from a person | treat it as a refusal |
| **Lazy loading** | 10 of 500 rows appear | content loads as you scroll | find the API underneath (section 1) |
| **Silent breakage** | a column quietly goes empty | the markup changed | check row counts on every run |
| **Terms of service** | `robots.txt` allows it; the ToS does not | two different documents | read both |

Four of these nine are the site saying no. **Recognising a refusal is a skill;
respecting it is a rule.**

---
# 3. Playwright and Scrapy (Optional)

| Tool | Use it when | Scale |
| --- | --- | --- |
| `requests` + BeautifulSoup | the page is static | dozens to hundreds of pages |
| **Selenium** | you need JavaScript or clicks; mature, widely documented | small |
| **Playwright** | the same need, with automatic waiting, faster runs, and the ability to capture the page's own API calls as it loads | small to medium |
| **Scrapy** | many pages or many sites — a framework with scheduling, retries, throttling and export built in | thousands and up |

**Playwright** is the modern alternative to Selenium.

---
# 4. Pagination

Almost no listing fits on one page. Recognise the pattern first; the code follows.

| Pattern | Looks like | Stop when |
| --- | --- | --- |
| Page number | `?page=2`, `page-2.html` | empty page or `404` |
| Next link | a "Next →" button | the link is gone |
| Offset | `?start=25&count=25` | fewer rows than `count` |
| Cursor | the API returns `next`, `has_next`, or a token | `has_next` is false |
| Infinite scroll | more rows as you scroll | find the API underneath |

**Follow the Next link** is the most general of the five, because you never build
URLs yourself — the page tells you where to go.

In [ ]:
def scrape_all(start_url, max_pages=5, delay=1.0):
    url, rows, page = start_url, [], 0
    while url and page < max_pages:            # two stopping rules
        page += 1
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        s = BeautifulSoup(resp.text, "html.parser")

        for card in s.select("article.product_pod"):
            rows.append({"title": card.h3.a["title"],
                         "price": card.select_one("p.price_color").text,
                         "page":  page})

        nxt = s.select_one("li.next a")
        url = urljoin(url, nxt["href"]) if nxt else None   # no link: stop
        print(f"page {page}: {len(rows)} rows so far")
        time.sleep(delay)
    return pd.DataFrame(rows)


books = scrape_all("https://books.toscrape.com/catalogue/page-1.html")

Two stopping rules working together. **The data's own rule** — no Next link
means the listing has ended. **Your safety cap** — `max_pages` — in case the
site never stops, or a bug sends you in circles. Never rely on only one.

---
# 5. The whole project, as pseudo-code

Every scraping project is a few loops nested inside each other. The outer loops
are decisions *you* make; the inner ones are mechanics.

```text
check robots.txt and the terms of service            # rule 1
seen = empty set

for SOURCE in your sources:                          # which site
    for KEYWORD in your keyword list:                # what to search for
        for FILTER in your filters:                  # city, date range, level
            url = first results page
            while url exists and pages < MAX:        # PAGINATION
                listing = fetch(url)
                for ITEM in listing:                 # each record on the page
                    if ITEM already in seen: skip    # deduplicate
                    detail = fetch(ITEM's link)      # one hop, no further
                    row = parse(detail)              # the fields you chose
                    save row + source + collected_at
                    add ITEM to seen
                    wait 1 second                    # rule 2
                url = the page's Next link, if any

write one line to the collection log                 # rule 8
```

Now match it to the manual process from week 2:

| Manual step | In the pseudo-code |
| --- | --- |
| Choose the source | `for SOURCE` |
| Write the keyword list | `for KEYWORD` |
| Search, then filter | `for FILTER` |
| Open each result | `for ITEM` and `fetch(ITEM's link)` |
| Copy the fields into a row | `parse` and `save` |
| Repeat, next page | `while url exists` |

**Nothing new happens.** The loops are the steps you would do by hand, nested in
the order you would do them.

---
# 6. Web data collection wrap-up

Scraping is one method among many. Pick by the question, not by the tool.

| Method | Examples | Kind of data | Your tools |
| --- | --- | --- | --- |
| **Survey / interview** | questionnaire; alumni interviews (Track 5) | **primary** | 问卷星, Qualtrics, a recorder, consent forms |
| **Experiment** — lab or field | A/B test, lab task, field trial | **primary** | randomisation, a control group |
| **Administrative records** | transactions, CRM, HR records | secondary | SQL (MGS 3501) |
| **Sensors and logs** | clickstream, app events, IoT readings | secondary | log pipelines, warehouses |
| **Web data** | see below | *found* data | APIs, feeds, scraping |

**Primary data** (一手数据) is collected by you, for your question — you control
the design and the sample. **Secondary data** was produced by someone else for
their own purpose, and you inherit their choices.

**Web data sits awkwardly in between.** You collect it yourself, but somebody else
generated it for their own reasons. Its bias comes from the platform — who posts,
what gets shown, what gets deleted — not from a sampling design you chose. Say so
in your methods section.

## Three kinds of web data

| | Examples | Usually reachable? |
| --- | --- | --- |
| **Organisational content** | company websites, job postings, programme pages | yes |
| **User-generated content** | reviews, posts, Q&A — 牛客网, Reddit, 大众点评 | often, with care over personal data |
| **User interaction data** | clicks, views, purchases | rarely — the platform keeps it |

## For web data: four routes, in this order

1. **An existing dataset** — Kaggle, open-data portals, university repositories.
   Check who collected it, when, and how.
2. **An official API** — week 2-2.
3. **A documented feed, sitemap or archive** — below.
4. **HTML scraping** — static first; then the page's hidden API; then browser
   automation.

**arXiv offers all four at once**, which makes it a good place to see the choice.
It publishes bulk metadata downloads (OAI-PMH, and full text via S3), an API for
querying, daily RSS feeds per subject, and the website itself. Which one you use
depends on the job: everything → bulk; a specific search → the API; *today's new
papers* → the feed.

A **feed** is a file the site publishes precisely so programs can read it. It is
XML, not HTML — so we use an XML parser.

In [16]:
FEED = "https://rss.arxiv.org/rss/cs.AI"       # arXiv's documented feed URL

resp = requests.get(FEED, headers=HEADERS, timeout=20)
resp.raise_for_status()
root = ET.fromstring(resp.content)


def local(tag):
    return tag.rsplit("}", 1)[-1]               # "{namespace}creator" -> "creator"


papers = []
for item in root.iter("item"):
    f = {local(ch.tag): (ch.text or "").strip() for ch in item}
    papers.append({
        "title":    f.get("title", ""),
        "authors":  f.get("creator", ""),
        "type":     f.get("announce_type", ""),
        "link":     f.get("link", ""),
        "subjects": "; ".join(c.text for c in item.findall("category") if c.text),
    })

feed = pd.DataFrame(papers)
print(f"{len(feed)} papers announced today in cs.AI")
if feed.empty:
    print("Empty — arXiv publishes no feed on Saturdays, Sundays or its holidays.")
feed.head()

214 papers announced today in cs.AI


,title,authors,type,link,subjects
0,RBS-Attention: Radius-Bounded Sparse Prefill f...,"Chuxu Song, Jiuqi Wei, Zhencan Peng",new,https://arxiv.org/abs/2609.20971,cs.AI
1,Attention-Aware Routing: Coupling Routing and ...,"Despoina Kosmopoulou, Anastasios Tsetsilas, Ef...",new,https://arxiv.org/abs/2609.20974,cs.AI
2,CaLR: Causal Latent Revision for Robust Diffus...,"Wei Cai, Jian Zhao, Yuchen Yuan, Xuelong Li",new,https://arxiv.org/abs/2609.20981,cs.AI
3,LoRA Enhanced Contrastive Learning with SAS Vi...,"Dan Zimmerman, Frank E. Bobe III, Amelia L. Mc...",new,https://arxiv.org/abs/2609.21061,cs.AI
4,Detecting Hallucination in LLMs: Tracing the T...,"Amir Jalilifard, Anderson Rocha, Eric Wong, Ma...",new,https://arxiv.org/abs/2609.21096,cs.AI; cs.CL; cs.LG


No pagination, no selectors, no JavaScript — and it is **permitted by design**.
Every field arrives labelled: `title`, `creator`, `category`, `announce_type`.

Two things from arXiv's own documentation worth copying into your habits: it asks
you to read its API terms of use before relying on the feed, and to acknowledge
it in anything you build. **A documented route comes with documented conditions.**

---
## Before Thursday

1. Place your own track's source on the four-route ladder. Which route does it
   allow, and why not the one above it?
2. If the answer is scraping, run the static test: is your field in the HTML, or
   does it need JavaScript?
3. Collect your pilot data with whichever route you chose.

> ⏰ **A1 — project brief and pilot data — due Monday 28 September.**